In [15]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from astroquery.gaia import Gaia
import pandas as pd
import plotly.io as pio

Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

In [16]:
!pip install nbformat --upgrade


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
t = Table.read('GalahC.fits')
df = t.to_pandas()
#Picking stars that fulfill parameters to be a giant
giants = df[(df['teff'] < 5500) & (df['logg'] < 3.0) & (df['flag_sp'] == 0)]
print(f"Total giants found: {len(giants)}")                                             #Print number of giants that fulfill condition
giants_sample = giants.sample(40, random_state=42)                                      #Choose 40 random giants from this selection
print(giants_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])       
giants_sample.to_csv('giants.csv', index=False)                                         

subgiants = df[(df['teff'].between(5000, 6500)) & (df['logg'].between(3.0, 4.0))]
print(f"Total subgiants found: {len(subgiants)}")
subgiants_sample = subgiants.sample(40, random_state=42)
print(subgiants_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
subgiants_sample.to_csv('subgiants.csv', index=False)

soba = df[(df['teff'] > 7500) & (df['logg'] > 3.5)]
print(f"Total soba found: {len(soba)}")
soba_sample = soba.sample(40, random_state=42)
print(soba_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
soba_sample.to_csv('soba.csv', index=False)

sfg = df[(`df['teff'].between (5200, 7500)) & (df['logg'] > 4.0)]
print(f"Total sfg found: {len(sfg)}")
sfg_sample = sfg.sample(40, random_state=42)
print(sfg_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
sfg_sample.to_csv('sfg.csv', index=False)

skm = df[(df['teff'] < 5200) & (df['logg'] > 4.0)]
print(f"Total skm found: {len(skm)}")
skm_sample = skm.sample(40, random_state=42)
print(skm_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
skm_sample.to_csv('skm.csv', index=False)

Total giants found: 196837
             sobject_id         teff      logg      fe_h    gaiadr3_source_id
890476  150719002901107  4897.139160  2.019355 -1.772829  2595741795476771200
871161  171102002101068  4753.502441  2.727344 -0.362278  2700927743579335552
800215  210927001601294  5070.670898  2.477721 -0.673982  6699358024880580992
639432  230804002101248  4531.378906  1.870785  0.247995  5940172630619093760
814237  210923002101111  4882.865723  2.785361 -0.341450  6469518278873255424
825154  170912001901374  4917.215332  2.534816 -0.351442  6888792982011753472
304273  230411000101012  4446.207031  1.861177 -0.322229  5410150026798168192
771474  160923001801198  4599.282227  2.508964  0.072870  4264556561051521920
639790  140610003901134  4884.726074  2.508054 -0.207141  5818379585873076224
663775  200826001201363  4742.697754  2.427677 -0.195302  5916270347357185536
686791  170530004101399  4849.792480  2.678227  0.208369  4123815357519951616
582600  210327004101297  5147.813965 

In [ ]:
# Query selected giants from the Gaia Catalogue, extract information and turn to csv
# Load your saved giants
giants = pd.read_csv('giants.csv')

# Get Gaia source IDs
source_ids = giants['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('giantsgaiadistance.csv', index=False)
print("Saved!")

     source_id              ra         ...     pm    distance_gspphot
                           deg         ...  mas / yr        pc       
------------------- ------------------ ... --------- ----------------
2700927743579335552  327.6124392353991 ...  8.436735         958.2991
4065921981507507968 270.92316376456733 ...  9.513159               --
3472564900874220416 182.06785578971161 ...  14.29287               --
4805875134992605056  81.85068748928441 ...  8.090865        1437.4958
6361755625270942080 299.68894857760745 ...  9.526618               --
6719483279571823872 271.05359888252985 ...  7.674997        3594.2617
5439386728166692736 145.25189308800856 ...  8.536294               --
6430349139206212096 301.51565542503937 ... 11.722585        3689.1514
5297666482574966144 130.91343765308238 ... 3.8820226               --
                ...                ... ...       ...              ...
4264556561051521920  290.7421711490286 ...  3.687307         2126.807
4709517578066091008 

In [19]:
# Convert giants given ra, dec and distance into cartesian
df = pd.read_csv('giantsgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total giants:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('giantscartesian.csv', index=False)

Total giants:           40
With complete data:     24
Missing distance:       16
              source_id          ra        dec  distance_gspphot         x_pc  \
0   2700927743579335552  327.612439   7.732924          958.2991   801.871047   
3   4805875134992605056   81.850687 -42.328477         1437.4958   150.646238   
5   6719483279571823872  271.053599 -47.083513         3594.2617    45.002993   
7   6430349139206212096  301.515655 -63.554606         3689.1514   858.818592   
9   4472051619419453312  273.006759   6.868843         1885.4786    98.190594   
10  6719780216435046656  270.874973 -46.221645         7053.0967    74.517715   
11  4471620645221095552  272.845161   5.521977         3187.3700   157.477339   
12  5916270347357185536  260.240426 -57.526211         1021.4010   -92.962291   
13  5818379585873076224  252.908476 -62.382927         2017.8293  -274.908681   
14  6699358024880580992  304.524452 -33.966403         2414.1040  1134.748013   
17  5410150026798168192  144

In [20]:
# Query selected subgiants from the Gaia Catalogue, extract information and turn to csv
# Load your saved subgiants
subgiants = pd.read_csv('subgiants.csv')

# Get Gaia source IDs
source_ids = subgiants['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('subgiantsgaiadistance.csv', index=False)
print("Saved!")

     source_id              ra         ...     pm    distance_gspphot
                           deg         ...  mas / yr        pc       
------------------- ------------------ ... --------- ----------------
3279149325099558528  67.53389605155388 ...  8.853182               --
6362633550946333952 300.36150720754165 ... 27.484438         836.3078
5778479378346285824 233.97015408296872 ... 21.119394        1103.7737
5282033110852778496 104.66371129009885 ... 7.2172823               --
5247346577015672960 142.63666077483876 ... 18.175438         394.5865
3632680113437172736   204.880264803821 ... 20.761595        2426.9058
 651971263827034752 124.73195787725142 ... 4.2202463               --
2637825225073056256 349.26214027378666 ... 7.0202804         737.0787
2668528571882415488 329.72802974663904 ...  5.938703               --
                ...                ... ...       ...              ...
5280394494930242944 101.36222893061102 ... 13.679063               --
5519611494581067392 

In [21]:
# Convert subgiants given ra, dec and distance into cartesian
df = pd.read_csv('subgiantsgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total subgiants:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('subgiantscartesian.csv', index=False)

Total subgiants:           40
With complete data:     22
Missing distance:       18
              source_id          ra        dec  distance_gspphot         x_pc  \
1   6362633550946333952  300.361507 -78.452486          836.3078    84.619355   
2   5778479378346285824  233.970154 -79.876094         1103.7737  -114.123004   
4   5247346577015672960  142.636661 -67.210743          394.5865  -121.477872   
5   3632680113437172736  204.880265  -5.151592         2426.9058 -2192.768850   
7   2637825225073056256  349.262140  -2.215489          737.0787   723.631060   
9   3285020515329552640   65.676251   6.119164         1211.3047   496.084117   
12  4709293861809114240   17.776810 -63.582674         2341.9282   992.188597   
13  5751038007655559296  132.144112  -7.794765         1435.2880  -954.176316   
17  4267765004718864640  288.578227   1.679688         1613.0650   513.700272   
18  5359485424343079552  164.820643 -52.670656          817.1020  -478.200053   
21  6662384506857513344  

In [22]:
# Query selected soba from the Gaia Catalogue, extract information and turn to csv
# Load your saved soba
soba = pd.read_csv('soba.csv')

# Get Gaia source IDs
source_ids = soba['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('sobagaiadistance.csv', index=False)
print("Saved!")

     source_id              ra         ...     pm     distance_gspphot
                           deg         ...  mas / yr         pc       
------------------- ------------------ ... ---------- ----------------
5254556074797382016  159.1194719477569 ...  4.8974514        2082.3887
5816308453873136384 251.51669131593752 ...   4.991061        1445.1338
5422652985827886080  140.9921985524466 ... 13.4738655         772.2417
5324336992532447104 136.26111658387393 ...  12.771995               --
5203848866226296448 144.06529084845738 ...   8.949338               --
4290354917755176832 296.66955494002775 ...  6.4795856          595.956
5847026124383884928  210.0886602159895 ...   8.871648               --
4154625533092731648  278.6248533087518 ...  0.7545518        1630.5806
4471093940487662592 271.99927178808747 ...    7.22137         783.3102
                ...                ... ...        ...              ...
5370839050126294016 177.98538013708728 ...  4.7866426         3053.963
578195

In [23]:
# Convert soba given ra, dec and distance into cartesian
df = pd.read_csv('sobagaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total soba:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('sobacartesian.csv', index=False)

Total soba:           40
With complete data:     23
Missing distance:       17
              source_id          ra        dec  distance_gspphot         x_pc  \
0   5254556074797382016  159.119472 -59.175083         2082.3887  -996.972236   
1   5816308453873136384  251.516691 -64.140573         1445.1338  -199.828156   
2   5422652985827886080  140.992199 -46.821265          772.2417  -410.619510   
5   4290354917755176832  296.669555   4.388870          595.9560   266.707028   
7   4154625533092731648  278.624853 -11.428259         1630.5806   239.680622   
8   4471093940487662592  271.999272   4.284455          783.3102    27.250815   
11  5902773326741267456  230.038621 -47.977663         1319.7517  -567.426316   
13  3086764137756140672  115.424384   0.778607          783.8027  -336.470741   
15  5219345997284525824  142.332739 -70.795541          657.1332  -171.104392   
17  6732230914304634624  282.810697 -35.077640          731.1035   132.664962   
18  4204875580969373824  289.3

In [24]:
# Query selected skm from the Gaia Catalogue, extract information and turn to csv
# Load your saved skm
skm = pd.read_csv('skm.csv')

# Get Gaia source IDs
source_ids = skm['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('skmgaiadistance.csv', index=False)
print("Saved!")

     source_id              ra         ...     pm    distance_gspphot
                           deg         ...  mas / yr        pc       
------------------- ------------------ ... --------- ----------------
 598878213846246400  132.4851887388852 ... 57.712067          168.952
6245767284665912192  242.6439127730848 ...  34.36065          295.307
3217591384209240832    84.170120200841 ... 4.9752812         635.4495
6697361243045768320  300.7962504517758 ... 6.3801255               --
6699035730535132288 300.58300512858136 ... 24.273653          354.373
  18045704526617600 39.287619785436725 ... 13.905776         538.2892
3810578964507624192 168.62461280997903 ...   76.3038         300.0752
6912217978458229120  316.0746904047655 ... 20.439913         134.1201
  38294551183787264  59.21261556423229 ... 110.87394         173.7014
                ...                ... ...       ...              ...
6249770189887222400 243.58526477534542 ...  25.29945         187.3882
2559186882144155520 

In [25]:
# Convert skm given ra, dec and distance into cartesian
df = pd.read_csv('skmgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total skm:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('skmcartesian.csv', index=False)

Total skm:           40
With complete data:     30
Missing distance:       10
              source_id          ra        dec  distance_gspphot        x_pc  \
0    598878213846246400  132.485189  11.235466          168.9520 -111.923162   
1   6245767284665912192  242.643913 -19.200927          295.3070 -128.150433   
2   3217591384209240832   84.170120  -1.157187          635.4495   64.532692   
4   6699035730535132288  300.583005 -33.763951          354.3730  149.889619   
5     18045704526617600   39.287620   5.222301          538.2892  414.894102   
6   3810578964507624192  168.624613   1.945696          300.0752 -294.010912   
7   6912217978458229120  316.074690  -4.975143          134.1201   96.235351   
8     38294551183787264   59.212616  12.330909          173.7014   86.858607   
9   5539888962894146816  124.230525 -39.598697          340.9858 -147.796985   
10   651621000652931072  127.162125  14.062148          364.8187 -213.772832   
11  6322261339399363200  229.016438  -7.10

In [26]:
# Query selected sfg from the Gaia Catalogue, extract information and turn to csv
# Load your saved sfg
sfg = pd.read_csv('sfg.csv')

# Get Gaia source IDs
source_ids = sfg['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('sfggaiadistance.csv', index=False)
print("Saved!")

     source_id              ra         ...     pm    distance_gspphot
                           deg         ...  mas / yr        pc       
------------------- ------------------ ... --------- ----------------
4388823502631790592  262.0185529022974 ... 4.5713525         559.1291
6358399434746290688   328.048717158088 ... 21.298218         519.3676
3463914802380464896 178.36379289449454 ...  33.83405         653.9867
6078528061394871552 190.46631932856315 ... 24.861292               --
2682024115040749568 331.76048998398727 ... 13.428739         318.3324
5254458729322793856 159.11183900783175 ... 17.525051         819.1204
5910934898452327168  261.1506716297253 ...  9.841678         862.8665
3204830383537431296  67.12172179777284 ...  7.133768         607.4482
4201315950739281024 289.64771779471613 ...  8.818415          839.655
                ...                ... ...       ...              ...
4901009588310956800   8.78456172403076 ...  10.28445         637.6227
5384390496573059328 

In [27]:
# Convert sfg given ra, dec and distance into cartesian
df = pd.read_csv('sfggaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total skm:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('sfgcartesian.csv', index=False)

Total skm:           40
With complete data:     35
Missing distance:       5
              source_id          ra        dec  distance_gspphot        x_pc  \
0   4388823502631790592  262.018553   3.185285          559.1291  -77.516494   
1   6358399434746290688  328.048717 -74.993698          519.3676  114.103858   
2   3463914802380464896  178.363793 -36.459422          653.9867 -525.772602   
4   2682024115040749568  331.760490   0.976143          318.3324  280.402945   
5   5254458729322793856  159.111839 -60.000663          819.1204 -382.635483   
6   5910934898452327168  261.150672 -60.920430          862.8665  -64.515010   
7   3204830383537431296   67.121722  -2.962522          607.4482  235.844867   
8   4201315950739281024  289.647718  -9.301314          839.6550  278.610293   
10  2609955865188371456  343.178381  -8.708024          533.9932  505.251719   
11  5886491174425620608  227.948659 -56.470407          439.2200 -162.499870   
12  6112541728474832128  203.342454 -42.865

In [28]:
# Load all your star type CSVs
giants      = pd.read_csv('giantscartesian.csv')
subgiants      = pd.read_csv('subgiantscartesian.csv')
mainsoba = pd.read_csv('sobacartesian.csv')
mainskm = pd.read_csv('skmcartesian.csv')
mainsfg = pd.read_csv('sfgcartesian.csv')

# Label each group
giants['region']      = 'Giants'
subgiants['region'] = 'Subgaints'
mainsoba['region'] = 'Main Sequence O/B/A'
mainskm['region'] = 'Main Sequence K/M'
mainsfg['region'] = 'Main Sequence F/G'

# Combine into one dataframe
all_stars = pd.concat([giants, subgiants, mainsoba, mainskm, mainsfg], ignore_index=True)

# Define colours per type
colours = {
    'Giants':   'red',
    'Subgiants': 'orange',
    'Main Sequence O/B/A': 'white',
    'Main Sequence F/G': 'yellow',
    'Main Sequence K/M': 'brown',
}

# Plot
fig = go.Figure()

for region, group in all_stars.groupby('region'):
    # Drop rows missing cartesian coords
    group = group.dropna(subset=['x_pc', 'y_pc', 'z_pc'])
    
    fig.add_trace(go.Scatter3d(
        x=group['x_pc'],
        y=group['y_pc'],
        z=group['z_pc'],
        mode='markers',
        name=region,
        marker=dict(
            size=4,
            color=colours.get(region, 'grey'),
            opacity=0.8
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'x: %{x:.1f} pc<br>'
            'y: %{y:.1f} pc<br>'
            'z: %{z:.1f} pc'
        ),
        text=group['source_id']
    ))

# Add Sun at origin (since we used ICRS/Earth centred coords)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=8, color='yellow', symbol='diamond'),
    name='Earth'
))

fig.update_layout(
    title='3D Star Positions',
    scene=dict(
        bgcolor='black',
        xaxis=dict(backgroundcolor='black', gridcolor='grey', title='X (pc)'),
        yaxis=dict(backgroundcolor='black', gridcolor='grey', title='Y (pc)'),
        zaxis=dict(backgroundcolor='black', gridcolor='grey', title='Z (pc)'),
    ),
    paper_bgcolor='black',
    font_color='white'
)

fig.show()